# EdgeLogic-AOI — Kaggle Full Training (70 Epochs + Deep Evaluation)

Notebook tự động nhận diện đường dẫn dữ liệu trên **Kaggle** (hoặc Colab), train chuẩn **70 epochs (`EfficientAD-M` & `PatchCore wide_resnet50_2`)**, chạy luôn `evaluate.py` bóc tách số liệu CV và đóng gói toàn bộ `checkpoints + reports` vào 1 file `.zip` để tải về máy.

In [ ]:
# CELL 1: Kiem tra GPU & Cai thu vien can thiet
!nvidia-smi
!pip install -q anomalib==2.7.1 "pandas<3.0" mlflow pyyaml scikit-learn opencv-python-headless

In [ ]:
# CELL 2: Nap source code EdgeLogic-AOI (Tu dong tim tu Kaggle Input zip HOAC git clone)
import os, shutil, glob
from pathlib import Path

WORK_DIR = "/kaggle/working/EdgeLogic-AOI"
OUT_DIR = "/kaggle/working/outputs"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Cach 1: Neu ban upload dataset code (edgelogic_code.zip) vao Kaggle Input
found_train = list(Path("/kaggle/input").rglob("python_research/training/train.py"))
if found_train:
    src_root = found_train[0].parents[2]
    print(f"[OK] Copy code tu Kaggle Input: {src_root}")
    shutil.copytree(src_root, WORK_DIR, dirs_exist_ok=True)
else:
    # Cach 2: Neu ban push code len GitHub (dien link repo cua ban vao day)
    REPO_URL = "https://github.com/<YOUR_USERNAME>/edgelogic-aoi.git"
    if "<YOUR_USERNAME>" not in REPO_URL:
        !git clone {REPO_URL} {WORK_DIR}
    else:
        raise FileNotFoundError(
            "Chua tim thay source code! Hay upload file 'edgelogic_code.zip' vao '+ Add Input' "
            "tren Kaggle HOAC dien link GitHub vao REPO_URL o tren."
        )

%cd {WORK_DIR}
!ls -la

In [ ]:
# CELL 3: Tu dong tim duong dan MVTec AD & MVTec LOCO tren Kaggle (va giai nen .tar.xz neu can)
from pathlib import Path
import subprocess

EXTRACT_ROOT = Path("/kaggle/working/data")
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

# Neu ban upload file .tar.xz len Kaggle Dataset, tu dong giai nen ra SSD /kaggle/working/data
for archive in Path("/kaggle/input").rglob("*.tar.xz"):
    cat_name = archive.name.replace(".tar.xz", "")
    target_sub = "mvtec_loco" if cat_name in ["splicing_connectors", "screw_bag"] else "mvtec_ad"
    dest = EXTRACT_ROOT / target_sub
    dest.mkdir(parents=True, exist_ok=True)
    if not (dest / cat_name / "train").exists():
        print(f">>> Giai nen {archive.name} -> {dest} ...")
        subprocess.run(["tar", "-xf", str(archive), "-C", str(dest)], check=True)

def find_dataset_root(sample_category: str) -> str | None:
    for search_base in [EXTRACT_ROOT, Path("/kaggle/input")]:
        matches = list(search_base.rglob(f"{sample_category}/train/good"))
        if matches:
            # Cha cua <category>/train/good chinh la dataset root
            return str(matches[0].parents[2])
    return None

MVTEC_AD_ROOT = find_dataset_root("screw") or find_dataset_root("metal_nut")
MVTEC_LOCO_ROOT = find_dataset_root("splicing_connectors") or find_dataset_root("screw_bag")

print("MVTEC_AD_ROOT   :", MVTEC_AD_ROOT)
print("MVTEC_LOCO_ROOT :", MVTEC_LOCO_ROOT)

In [ ]:
# CELL 4: Train Chuan Bai Bao (70 Epochs, EfficientAD Medium + Small) & Danh gia Sau
# Moi class ~35-50 phut tren Kaggle GPU P100/T4. Co the comment bot de chay theo dot (tranh qua 12h/session).

EPOCHS = 70
IMAGENET_DIR = "/kaggle/working/imagenette"
CKPT_OUT = f"{OUT_DIR}/checkpoints"

JOBS = [
    # (dataset, root, category, model_size)
    ("mvtec_loco", MVTEC_LOCO_ROOT, "splicing_connectors", "medium"),
    ("mvtec_loco", MVTEC_LOCO_ROOT, "screw_bag", "medium"),
    ("mvtec_ad",   MVTEC_AD_ROOT,   "screw", "small"),
    ("mvtec_ad",   MVTEC_AD_ROOT,   "metal_nut", "small"),
    ("mvtec_ad",   MVTEC_AD_ROOT,   "transistor", "small"),
]

for ds_name, ds_root, cat, msize in JOBS:
    if not ds_root or not (Path(ds_root) / cat).exists():
        print(f"[SKIP] Chua co du lieu cho {ds_name}/{cat}")
        continue

    !python python_research/training/train.py \
        --model efficientad --model-size {msize} \
        --dataset {ds_name} --category {cat} \
        --epochs {EPOCHS} --num-workers 2 \
        --imagenet-dir {IMAGENET_DIR} \
        --data-root {ds_root} --output-dir {CKPT_OUT}

    # Tim checkpoint vua train xong va chay evaluate.py boc tach 4 chi so cong nghiep
    ckpts = sorted((Path(CKPT_OUT) / "efficientad" / ds_name / cat).rglob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
    if ckpts:
        best_ckpt = str(ckpts[0])
        !python python_research/training/evaluate.py \
            --model efficientad --dataset {ds_name} --category {cat} \
            --ckpt "{best_ckpt}" --data-root {ds_root}

In [ ]:
# CELL 5: Dong goi toan bo Checkpoints (.ckpt) + Bao cao JSON de tai 1 file duy nhat ve Laptop!
!cp -r artifacts/reports {OUT_DIR}/reports
!cd {OUT_DIR} && zip -r /kaggle/working/edgelogic_trained_bundle.zip checkpoints reports
!ls -lh /kaggle/working/edgelogic_trained_bundle.zip
print("[DONE] Tai file /kaggle/working/edgelogic_trained_bundle.zip o tab Output ve may!")